# The Particle Filter — SIS, Degeneracy & Resampling

*Course 4 — Particle Filters, Part 2. Assembling the [Bayesian recursion, Monte-Carlo integration, importance sampling, and impulse representation](15_Bayesian_Recursion_MonteCarlo_Importance_Sampling.ipynb) into the **particle filter**: first the basic SIS algorithm, then the degeneracy problem, then the **resampling** fix that makes it work.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 Sequential Importance Sampling (SIS)

- Represent the posterior as $N_p$ weighted particles and update them recursively. With an importance density chosen to factor conveniently, a new particle is drawn and its weight updated by:

$$
x_k^{(i)} \sim q\big(x_k \mid x_{k-1}^{(i)}, z_k\big), \qquad
\tilde{w}_k^{(i)} = \frac{f\big(z_k \mid x_k^{(i)}\big)\, f\big(x_k^{(i)} \mid x_{k-1}^{(i)}\big)}{q\big(x_k^{(i)}\mid x_{k-1}^{(i)}, z_k\big)}\,\tilde{w}_{k-1}^{(i)}.
$$

  → For each particle: **propose** a new state from $q$, then **multiply its weight** by (measurement likelihood × transition prior ÷ proposal). Likely-looking particles gain weight; unlikely ones lose it. Then normalize $w_k^{(i)} = \tilde w_k^{(i)}/\sum_j \tilde w_k^{(j)}$.

- **→ Intuition:** each particle is one hypothesized state trajectory; its weight is "how consistent this hypothesis is with the measurements so far." The state estimate is the weighted mean $\hat x_k = \sum_i w_k^{(i)} x_k^{(i)}$.

### 🧩 A Convenient Importance Density

- The natural, simplest choice is the **transition prior** $q\big(x_k\mid x_{k-1}^{(i)},z_k\big) = f\big(x_k\mid x_{k-1}^{(i)}\big)$ — just propagate each particle through the process model.

- Then the weight update **collapses**:

$$
\tilde{w}_k^{(i)} = f\big(z_k \mid x_k^{(i)}\big)\,\tilde{w}_{k-1}^{(i)}.
$$

  → The transition and proposal terms cancel, leaving weight $\propto$ **measurement likelihood**. So: push each particle forward by the dynamics (with noise), then weight it by how well it explains the measurement. This is the **bootstrap / SIR** filter.

- **→ Intuition:** "simulate many possible futures, keep the ones the sensor agrees with." Beautifully simple — after all the heavy math, the algorithm is just *propagate and reweight*.

```octave
for i = 1:Np
  xp(i,k) = f(xp(i,k-1)) + processNoise;      % propose from prior
  wtilde(i) = w(i,k-1) * likelihood(z(k), xp(i,k));  % weight by f(z|x)
end
w(:,k) = wtilde / sum(wtilde);                % normalize
xhat(k) = sum(w(:,k) .* xp(:,k));             % state estimate
```

### 🧩 The Degeneracy Problem

- Run plain SIS and something bad happens: after a few steps **one particle takes almost all the weight** and the rest decay to ~0.

  → We are drawing high-dimensional trajectories at random; it's exponentially unlikely to keep proposing good ones. Almost all weight concentrates on a single trajectory — the state estimate then depends on **one particle**, wasting all the others' computation. This is **degeneracy**.

- Diagnose it with the **effective sample size**:

$$
N_{\text{eff},k} = \frac{1}{\sum_{i=1}^{N_p}\big(w_k^{(i)}\big)^{2}}.
$$

  → If all weights are equal, $N_{\text{eff}} = N_p$ (healthy); if one weight is 1 and the rest 0, $N_{\text{eff}} = 1$ (fully degenerate). It measures how many particles are *actually contributing*.

- **→ Intuition:** degeneracy is the particle filter's version of "all my eggs in one basket." Adding more particles only delays it — we need to actively cull the useless ones.

### 🧩 Resampling — the Fix

- When $N_{\text{eff}}$ drops below a threshold (e.g. a fraction of $N_p$), **resample**: draw $N_p$ new particles from the current set **with replacement, with probability $\propto w^{(i)}$**, then reset all weights to $1/N_p$.

  → Delete low-weight particles and **duplicate high-weight ones**, concentrating computational effort where the probability is. High-probability particles spawn many copies; unlikely ones vanish.

- A simple **systematic resampling** algorithm using the cumulative weights $c^{(i)} = \sum_{j\le i} w^{(j)}$:

$$
u_1 \sim \mathcal{U}\big(0, N_p^{-1}\big), \quad u_j = u_1 + \frac{j-1}{N_p}; \quad \text{pick particle } i \text{ where } c^{(i-1)} < u_j \le c^{(i)}.
$$

  → March a single evenly-spaced comb $\{u_j\}$ up the cumulative-weight "staircase"; each step lands on a particle, choosing it once per unit of weight. Low-variance and $\mathcal{O}(N_p)$.

- **→ Intuition:** if there's process noise, the duplicated particles **diverge again** next step (they get different noise), restoring diversity. Resampling trades "many distinct low-weight particles" for "fewer distinct but high-weight particles that spread back out." (Resample too often and you lose diversity — hence the $N_{\text{eff}}$ threshold.)

```octave
nEff = 1/sum(w(:,k).^2);
if nEff < nEffThreshold
  c = cumsum(w(:,k));  u1 = rand/Np;  i = 1;
  for j = 1:Np
    uj = u1 + (j-1)/Np;
    while uj > c(i), i = i + 1; end
    xpNew(:,j) = xp(:,i);           % replicate particle i
  end
  xp(:,k) = xpNew;  w(:,k) = ones(Np,1)/Np;   % reset weights
end
```

### 🧩 SIS with Resampling — the Complete Particle Filter

Per time step:

1. **Propose:** draw $x_k^{(i)} \sim q(x_k\mid x_{k-1}^{(i)}, z_k)$ (prior model) for every particle.
2. **Weight:** $\tilde{w}_k^{(i)} = f(z_k\mid x_k^{(i)})\,\tilde{w}_{k-1}^{(i)}$; then normalize.
3. **Estimate:** $\hat{x}_k = \sum_i w_k^{(i)} x_k^{(i)}$ (and any percentiles/bounds from the particle set).
4. **Resample** if $N_{\text{eff},k} < $ threshold; reset weights to $1/N_p$.

- **→ Intuition:** with resampling, the particle filter closely matches the true Bayesian (brute-force) posterior — near-optimal state estimates *and* confidence bounds — while beating the curse of dimensionality. Without resampling it degenerates within ~20 steps; with it, the effective sample size and weight spread stay healthy and estimates track truth.

- **→ Practical notes:** more particles → better accuracy but more cost; the filter shines exactly where Kalman filters struggle — **highly nonlinear, non-Gaussian, or multimodal** posteriors. It is more expensive than an EKF/SPKF, so use it when that extra generality is actually needed.

### 🧩 Summary

- The **SIS** particle filter represents the posterior as weighted particles and updates each by **propose (from $q$) → reweight (by likelihood)**; choosing $q=$ prior gives the simple **bootstrap** filter with $\tilde w_k^{(i)} = f(z_k\mid x_k^{(i)})\tilde w_{k-1}^{(i)}$.

- Plain SIS suffers **degeneracy**: weight collapses onto one particle, measured by the **effective sample size** $N_{\text{eff}} = 1/\sum_i (w^{(i)})^2$.

- **Resampling** (draw with replacement $\propto$ weight, reset weights) when $N_{\text{eff}}$ is low kills degenerate particles and multiplies good ones; process noise re-diversifies them.

- The result approximates the **exact Bayesian posterior** for arbitrary nonlinear / non-Gaussian problems, at the cost of many function evaluations.

---
*Next: [17 · Particle-Filter Navigation Application](17_Particle_Filter_Navigation_Application.ipynb).*